# FraudShield Advanced: Feature Store SDK

## Concept Review: Feature Store Architecture

Amazon SageMaker Feature Store provides a centralized repository for ML features that serves
both training and real-time inference workloads through a **dual-store architecture**:

- **Online Store (DynamoDB-backed)**: Delivers single-digit millisecond lookups for the
  latest feature values. This is the store that serves real-time inference endpoints when
  a model needs fresh feature vectors at prediction time.

- **Offline Store (S3 + Glue Catalog)**: Persists the full history of feature records as
  Parquet files in S3, registered in the AWS Glue Data Catalog. This enables time-travel
  queries, batch feature retrieval for training datasets, and point-in-time joins that
  prevent data leakage.

Both stores are fed from a single `FeatureGroup` definition, ensuring schema consistency
across training and serving. In this notebook we will create a feature group for the
FraudShield transaction domain, ingest records, and query both stores.

**Environment**: SageMaker Studio on `ml.t3.medium` | Free Tier training on `ml.m5.xlarge`

In [ ]:
%pip install -q sagemaker boto3 pandas

In [ ]:
import time
import json
import pandas as pd
import numpy as np
import boto3
import sagemaker
from sagemaker.session import Session
from sagemaker.feature_store.feature_definition import (
    FeatureDefinition,
    FeatureTypeEnum,
)
from sagemaker.feature_store.feature_group import FeatureGroup

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session.boto_region_name
bucket = sagemaker_session.default_bucket()
prefix = "fraudshield-feature-store"

featurestore_client = boto3.client("sagemaker-featurestore-runtime", region_name=region)
sm_client = boto3.client("sagemaker", region_name=region)

print(f"Role:    {role}")
print(f"Region:  {region}")
print(f"Bucket:  {bucket}")

## Concept Review: Feature Groups

A **Feature Group** is the fundamental organizational unit in Feature Store. Each group:

1. **Has a schema** defined by a list of `FeatureDefinition` objects, each specifying a
   feature name and type (`Integral`, `Fractional`, or `String`).

2. **Requires a `record_identifier_feature_name`** -- the primary key used to look up
   individual records in the online store (e.g., `transaction_id`).

3. **Requires an `event_time_feature_name`** -- a timestamp that Feature Store uses to
   order records chronologically. This is critical for point-in-time correctness:
   the online store keeps the record with the most recent event time, while the offline
   store retains the full history.

When both `online_store_config` and `offline_store_config` are enabled, every ingested
record flows into DynamoDB immediately and is asynchronously written to S3 (typically
within 15 minutes).

In [ ]:
feature_group_name = f"fraudshield-transactions-{int(time.time())}"

feature_definitions = [
    FeatureDefinition(feature_name="transaction_id", feature_type=FeatureTypeEnum.STRING),
    FeatureDefinition(feature_name="event_time", feature_type=FeatureTypeEnum.STRING),
    FeatureDefinition(feature_name="amount", feature_type=FeatureTypeEnum.FRACTIONAL),
    FeatureDefinition(feature_name="hour", feature_type=FeatureTypeEnum.INTEGRAL),
    FeatureDefinition(feature_name="distance_from_home", feature_type=FeatureTypeEnum.FRACTIONAL),
    FeatureDefinition(feature_name="merchant_risk_score", feature_type=FeatureTypeEnum.FRACTIONAL),
    FeatureDefinition(feature_name="is_international", feature_type=FeatureTypeEnum.INTEGRAL),
    FeatureDefinition(feature_name="transaction_velocity_1h", feature_type=FeatureTypeEnum.INTEGRAL),
    FeatureDefinition(feature_name="card_age_days", feature_type=FeatureTypeEnum.INTEGRAL),
    FeatureDefinition(feature_name="is_fraud", feature_type=FeatureTypeEnum.INTEGRAL),
]

print(f"Feature group name: {feature_group_name}")
print(f"Number of features: {len(feature_definitions)}")
for fd in feature_definitions:
    print(f"  {fd.feature_name:30s}  {fd.feature_type.value}")

In [ ]:
feature_group = FeatureGroup(
    name=feature_group_name,
    sagemaker_session=sagemaker_session,
    feature_definitions=feature_definitions,
)

feature_group.create(
    s3_uri=f"s3://{bucket}/{prefix}/{feature_group_name}",
    record_identifier_name="transaction_id",
    event_time_feature_name="event_time",
    role_arn=role,
    enable_online_store=True,
)

print(f"Creating feature group '{feature_group_name}' with Online + Offline stores...")

In [ ]:
def wait_for_feature_group(fg, max_wait=300, poll_interval=15):
    """Poll until the feature group reaches 'Created' status."""
    elapsed = 0
    while elapsed < max_wait:
        status = fg.describe().get("FeatureGroupStatus")
        print(f"  Status: {status} ({elapsed}s elapsed)")
        if status == "Created":
            return
        if status == "CreateFailed":
            raise RuntimeError(fg.describe().get("FailureReason", "Unknown failure"))
        time.sleep(poll_interval)
        elapsed += poll_interval
    raise TimeoutError(f"Feature group not ready after {max_wait}s")

wait_for_feature_group(feature_group)
print("Feature group is Active.")

## Concept Review: Online Store

The **Online Store** is a managed, low-latency key-value store backed by DynamoDB. Key
behaviors:

- **Single-digit millisecond reads** via `get_record()` on the `sagemaker-featurestore-runtime` client.
- Stores only the **latest value** per record identifier. If you `put_record()` with the
  same `transaction_id` but a newer `event_time`, the previous value is overwritten in the
  online store (the offline store still retains both).
- Designed to be called from real-time inference endpoints, Lambda functions, or any
  application that needs fresh features at prediction time.

Below we demonstrate ingesting individual records and performing a low-latency lookup.

In [ ]:
from datetime import datetime, timezone

sample_records = [
    {
        "transaction_id": "txn-001",
        "event_time": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "amount": 149.99,
        "hour": 14,
        "distance_from_home": 5.2,
        "merchant_risk_score": 0.12,
        "is_international": 0,
        "transaction_velocity_1h": 2,
        "card_age_days": 730,
        "is_fraud": 0,
    },
    {
        "transaction_id": "txn-002",
        "event_time": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "amount": 4899.00,
        "hour": 3,
        "distance_from_home": 1240.5,
        "merchant_risk_score": 0.87,
        "is_international": 1,
        "transaction_velocity_1h": 8,
        "card_age_days": 45,
        "is_fraud": 1,
    },
]

for record in sample_records:
    featurestore_client.put_record(
        FeatureGroupName=feature_group_name,
        Record=[
            {"FeatureName": k, "ValueAsString": str(v)}
            for k, v in record.items()
        ],
    )
    print(f"Ingested {record['transaction_id']}")

print("Online store ingestion complete.")

In [ ]:
response = featurestore_client.get_record(
    FeatureGroupName=feature_group_name,
    RecordIdentifierValueAsString="txn-002",
)

print("Online Store lookup for txn-002:")
for feature in response["Record"]:
    print(f"  {feature['FeatureName']:30s}  {feature['ValueAsString']}")

## Concept Review: Offline Store

The **Offline Store** writes every ingested record to S3 as Apache Parquet files and
registers them in the AWS Glue Data Catalog. This enables:

- **Full historical access**: Every version of every record is retained, not just the latest.
- **SQL queries via Athena**: Because the Glue catalog provides a table interface, you can
  run standard SQL to filter, aggregate, or join feature data.
- **Time-travel queries**: Retrieve features as they existed at a specific point in time,
  which is essential for constructing leak-free training datasets.

Note: Records appear in the offline store asynchronously, typically within ~15 minutes
after ingestion. For this lecture we will also use `FeatureGroup.ingest()` to demonstrate
batch ingestion from a DataFrame.

In [ ]:
np.random.seed(42)
n_records = 50

batch_df = pd.DataFrame({
    "transaction_id": [f"txn-{i:04d}" for i in range(100, 100 + n_records)],
    "event_time": [datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")] * n_records,
    "amount": np.round(np.random.uniform(10, 5000, n_records), 2),
    "hour": np.random.randint(0, 24, n_records),
    "distance_from_home": np.round(np.random.uniform(0, 2000, n_records), 1),
    "merchant_risk_score": np.round(np.random.uniform(0, 1, n_records), 3),
    "is_international": np.random.randint(0, 2, n_records),
    "transaction_velocity_1h": np.random.randint(1, 15, n_records),
    "card_age_days": np.random.randint(1, 3650, n_records),
    "is_fraud": np.random.choice([0, 1], size=n_records, p=[0.9, 0.1]),
})

# Ensure correct dtypes for Feature Store
batch_df["amount"] = batch_df["amount"].astype(float)
batch_df["distance_from_home"] = batch_df["distance_from_home"].astype(float)
batch_df["merchant_risk_score"] = batch_df["merchant_risk_score"].astype(float)
batch_df["hour"] = batch_df["hour"].astype(int)
batch_df["is_international"] = batch_df["is_international"].astype(int)
batch_df["transaction_velocity_1h"] = batch_df["transaction_velocity_1h"].astype(int)
batch_df["card_age_days"] = batch_df["card_age_days"].astype(int)
batch_df["is_fraud"] = batch_df["is_fraud"].astype(int)

print(f"Batch DataFrame shape: {batch_df.shape}")
batch_df.head()

In [ ]:
ingestion_manager = feature_group.ingest(data_frame=batch_df, max_workers=3, wait=True)
print(f"Batch ingestion complete. Failed rows: {ingestion_manager.failed_rows}")

In [ ]:
fg_resolved_output_s3 = feature_group.describe().get("OfflineStoreConfig", {}).get(
    "S3StorageConfig", {}
).get("ResolvedOutputS3Uri", "Not available yet")

print(f"Offline store S3 location: {fg_resolved_output_s3}")
print("\nNote: Records appear in the offline store within ~15 minutes.")
print("The Athena query below may return 0 rows if run immediately after ingestion.")

# Build Athena query against the Glue catalog table
query_string = f"""
SELECT transaction_id, amount, is_fraud, event_time
FROM "{feature_group_name}"
WHERE is_fraud = 1
ORDER BY amount DESC
LIMIT 10
"""

print(f"Athena query:\n{query_string}")

# Execute via the Feature Store Athena helper
try:
    athena_query = feature_group.athena_query()
    athena_query.run(
        query_string=query_string.strip(),
        output_location=f"s3://{bucket}/{prefix}/athena-results/",
    )
    athena_query.wait()
    result_df = athena_query.as_dataframe()
    print(f"\nQuery returned {len(result_df)} rows:")
    display(result_df)
except Exception as e:
    print(f"Athena query not yet available (offline sync pending): {e}")

## Concept Review: Point-in-Time Queries

One of the most critical capabilities of the Offline Store is **point-in-time (PIT) correctness**.
When constructing a training dataset, you must ensure that the feature values used for each
training example reflect only information that was available **at the time the label event
occurred** -- not information from the future.

Without PIT joins, a model might train on features that incorporate future data, leading to
**data leakage** and inflated offline metrics that do not generalize to production.

The pattern involves:

1. Maintaining an **entity DataFrame** with the record identifier and the label event timestamp.
2. Joining against the offline store using the `event_time` column so that only feature
   records with `event_time <= label_event_time` are included.

SageMaker Feature Store's `create_dataset()` builder and Athena queries both support this
pattern natively.

In [ ]:
pit_query = f"""
SELECT t.transaction_id,
       t.amount,
       t.merchant_risk_score,
       t.is_fraud,
       t.event_time,
       t.write_time
FROM "{feature_group_name}" t
WHERE t.event_time <= '2026-03-24T23:59:59Z'
  AND NOT t.is_deleted
ORDER BY t.event_time DESC
LIMIT 20
"""

print("Point-in-time query pattern:")
print(pit_query)
print("This query retrieves features as they existed on or before the cutoff timestamp,")
print("excluding soft-deleted records. In production you would parameterize the cutoff")
print("to match each training example's label event time.")

try:
    athena_query = feature_group.athena_query()
    athena_query.run(
        query_string=pit_query.strip(),
        output_location=f"s3://{bucket}/{prefix}/athena-results/",
    )
    athena_query.wait()
    pit_df = athena_query.as_dataframe()
    print(f"\nReturned {len(pit_df)} records:")
    display(pit_df.head(10))
except Exception as e:
    print(f"Query not yet available (offline sync pending): {e}")

## Cleanup

Delete the feature group and remove S3 artifacts to avoid ongoing charges.
The online store is deleted immediately; the offline store S3 data must be
removed separately.

In [ ]:
feature_group.delete()
print(f"Deleted feature group: {feature_group_name}")

s3 = boto3.resource("s3")
s3_bucket = s3.Bucket(bucket)
objects = list(s3_bucket.objects.filter(Prefix=f"{prefix}/{feature_group_name}"))
if objects:
    s3_bucket.delete_objects(
        Delete={"Objects": [{"Key": obj.key} for obj in objects]}
    )
    print(f"Deleted {len(objects)} S3 objects under {prefix}/{feature_group_name}")

athena_objects = list(s3_bucket.objects.filter(Prefix=f"{prefix}/athena-results/"))
if athena_objects:
    s3_bucket.delete_objects(
        Delete={"Objects": [{"Key": obj.key} for obj in athena_objects]}
    )
    print(f"Deleted {len(athena_objects)} Athena result objects")

print("Cleanup complete.")